# Direction Extraction on Google Colab

Extract activation directions from language models.

**Direction Types:**
- **refusal**: Refusal direction (harmful vs benign, simple prompts)
- **search_clean**: Search direction (harmful vs benign, simple prompts + `<search>` token)

**Extraction Method:**
1. Load harmful and harmless prompts
2. For each prompt, get hidden state at last token
3. Compute direction: mean(harmful) - mean(benign)
4. Normalize and save

**Steps:**
1. Install dependencies
2. Configure settings (model, direction type, layers)
3. Upload datasets
4. Run extraction
5. Download results

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate matplotlib

## 2. Configuration

In [ ]:
# ============================================================================
# EDIT THIS CONFIGURATION
# ============================================================================

CONFIG = {
    # Model selection
    "model_preset": "qwen7b",  # Options: qwen3b, qwen7b, qwen14b, llama3b
    
    # Direction type - WHICH DIRECTION TO EXTRACT
    "direction_type": "refusal",  # Options: "refusal" or "search_clean"
    #   refusal      = Refusal direction (simple prompts, no <search>)
    #   search_clean = Search direction (simple prompts + <search>, NO system prompt)
    
    # Sampling
    "max_harmful": None,  # None = use all (~500)
    "max_harmless": 1000,
    "random_seed": 42,
}

# Model presets
MODEL_PRESETS = {
    "qwen3b": {
        "model_path": "Qwen/Qwen2.5-3B-Instruct",
        "layers": [14, 18, 22, 26, 28],
    },
    "qwen7b": {
        "model_path": "Qwen/Qwen2.5-7B-Instruct",
        "layers": [14, 18, 22, 26, 28],
    },
    "qwen14b": {
        "model_path": "Qwen/Qwen2.5-14B-Instruct",
        "layers": [24, 32, 40, 44, 48],
    },
    "llama3b": {
        "model_path": "meta-llama/Llama-3.2-3B-Instruct",
        "layers": [14, 18, 22, 26, 28],
    },
}

# Apply preset
CONFIG.update(MODEL_PRESETS[CONFIG["model_preset"]])

print("✓ Configuration loaded")
print(f"  Model: {CONFIG['model_path']}")
print(f"  Direction type: {CONFIG['direction_type']}")
print(f"  Layers: {CONFIG['layers']}")

## 3. Upload Datasets

In [ ]:
from google.colab import files
import json

print("Upload your datasets:")
print("  1. harmful_combined.json (harmful prompts)")
print("  2. harmless_val.json (harmless prompts)")
print()

uploaded = files.upload()

# Rename to expected names
for filename in uploaded.keys():
    if "harmful" in filename.lower():
        !mv "{filename}" /content/harmful.json
        print(f"✓ Harmful dataset: {filename}")
    elif "harmless" in filename.lower():
        !mv "{filename}" /content/harmless.json
        print(f"✓ Harmless dataset: {filename}")

CONFIG["harmful_path"] = "/content/harmful.json"
CONFIG["harmless_path"] = "/content/harmless.json"

## 4. Setup

In [ ]:
import torch
import numpy as np
import transformers
import random
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

random.seed(CONFIG["random_seed"])

print("✓ Imports complete")

## 5. Extraction Functions

In [ ]:
def prepare_prompt(instruction, tokenizer, direction_type):
    """
    Format instruction based on direction type.
    
    Args:
        direction_type: "refusal" or "search_clean"
    """
    question = instruction.strip()
    
    # Apply chat template
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": question}],
            add_generation_prompt=True,
            tokenize=False,
        )
    else:
        prompt = question
    
    # Add <search> token for search_clean direction
    if direction_type == "search_clean":
        prompt = prompt + "<search>"
    
    return prompt


def get_hidden_state(model, tokenizer, instruction, layer_idx, direction_type):
    """Get hidden state at last token for one instruction."""
    prompt = prepare_prompt(instruction, tokenizer, direction_type)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    
    # Last token (either last prompt token or <search> token)
    return outputs.hidden_states[layer_idx][0, -1, :].cpu()


print("✓ Extraction functions defined")

## 6. Load Datasets

In [ ]:
# Load datasets
with open(CONFIG["harmful_path"]) as f:
    harmful = [item["instruction"] for item in json.load(f)]

with open(CONFIG["harmless_path"]) as f:
    harmless = [item["instruction"] for item in json.load(f)]

# Sample if needed
if CONFIG["max_harmful"] and len(harmful) > CONFIG["max_harmful"]:
    harmful = random.sample(harmful, CONFIG["max_harmful"])

if CONFIG["max_harmless"] and len(harmless) > CONFIG["max_harmless"]:
    harmless = random.sample(harmless, CONFIG["max_harmless"])

print(f"✓ Loaded datasets:")
print(f"  Harmful: {len(harmful)}")
print(f"  Harmless: {len(harmless)}")

## 7. Load Model

In [ ]:
print(f"Loading model: {CONFIG['model_path']}")

tokenizer = transformers.AutoTokenizer.from_pretrained(
    CONFIG["model_path"],
    trust_remote_code=True
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    CONFIG["model_path"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print("✓ Model loaded")

## 8. Extract Directions

In [ ]:
print(f"Extracting {CONFIG['direction_type']} direction...")
print("=" * 80)

all_directions = {}
all_stats = {}

for layer_idx in tqdm(CONFIG["layers"], desc="Layers"):
    print(f"\nLayer {layer_idx}:")
    
    # Extract hidden states for harmful prompts
    print("  Extracting harmful...")
    harmful_states = []
    for inst in tqdm(harmful, desc="  Harmful", leave=False):
        state = get_hidden_state(model, tokenizer, inst, layer_idx, CONFIG["direction_type"])
        harmful_states.append(state)
    harmful_states = torch.stack(harmful_states)
    
    # Extract hidden states for harmless prompts
    print("  Extracting harmless...")
    harmless_states = []
    for inst in tqdm(harmless, desc="  Harmless", leave=False):
        state = get_hidden_state(model, tokenizer, inst, layer_idx, CONFIG["direction_type"])
        harmless_states.append(state)
    harmless_states = torch.stack(harmless_states)
    
    # Compute direction
    direction_raw = harmful_states.mean(dim=0) - harmless_states.mean(dim=0)
    direction_norm = direction_raw / direction_raw.norm()
    
    # Compute statistics
    harmful_projs = (harmful_states @ direction_norm)
    harmless_projs = (harmless_states @ direction_norm)
    separation = harmful_projs.mean().item() - harmless_projs.mean().item()
    
    print(f"  Separation: {separation:.4f}")
    print(f"  Direction norm: {direction_raw.norm().item():.1f}")
    
    # Store results
    all_directions[f"layer_{layer_idx}"] = direction_norm.float().numpy().tolist()
    all_stats[f"layer_{layer_idx}"] = {
        "separation": separation,
        "direction_norm": direction_raw.norm().item(),
        "harmful_proj_mean": harmful_projs.mean().item(),
        "harmful_proj_std": harmful_projs.std().item(),
        "harmless_proj_mean": harmless_projs.mean().item(),
        "harmless_proj_std": harmless_projs.std().item(),
    }

print("\n" + "=" * 80)
print("✓ Extraction complete!")

# Find best layer
best_layer = max(CONFIG["layers"], key=lambda l: all_stats[f"layer_{l}"]["separation"])
print(f"\nBest layer: {best_layer} (separation={all_stats[f'layer_{best_layer}']['separation']:.4f})")

## 9. Save and Download Results

In [ ]:
# Save direction
direction_file = f"/content/{CONFIG['direction_type']}_direction.json"
with open(direction_file, "w") as f:
    json.dump(all_directions, f, indent=2)

# Save stats
stats_file = f"/content/{CONFIG['direction_type']}_stats.json"
with open(stats_file, "w") as f:
    json.dump(all_stats, f, indent=2)

print(f"✓ Saved to {direction_file} and {stats_file}")

# Download
files.download(direction_file)
files.download(stats_file)
print("✓ Download started")

## 10. Visualize Results

In [ ]:
# Plot separation by layer
fig, ax = plt.subplots(figsize=(10, 6))

separations = [all_stats[f"layer_{l}"]["separation"] for l in CONFIG["layers"]]
colors = ['#2ecc71' if CONFIG["direction_type"] == "search_clean" else '#e74c3c']

ax.bar(range(len(CONFIG["layers"])), separations, color=colors, alpha=0.8)
ax.set_xticks(range(len(CONFIG["layers"])))
ax.set_xticklabels([f'Layer {l}' for l in CONFIG["layers"]])
ax.set_ylabel('Separation (harmful - benign)')
ax.set_title(f'{CONFIG["model_preset"].upper()}: {CONFIG["direction_type"].title()} Direction Separation by Layer')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/separation_by_layer.png', dpi=150, bbox_inches='tight')
plt.show()

# Download plot
files.download('/content/separation_by_layer.png')
print("✓ Plot saved and downloaded")

## 11. Summary Statistics

In [ ]:
print("Summary Statistics:")
print("=" * 80)
print(f"Model: {CONFIG['model_preset']}")
print(f"Direction type: {CONFIG['direction_type']}")
print(f"Harmful prompts: {len(harmful)}")
print(f"Harmless prompts: {len(harmless)}")
print()
print("Separation by layer:")
for layer in CONFIG["layers"]:
    stats = all_stats[f"layer_{layer}"]
    print(f"  Layer {layer:2d}: {stats['separation']:+.4f} (norm={stats['direction_norm']:.1f})")
print()
print(f"Best layer: {best_layer} (separation={all_stats[f'layer_{best_layer}']['separation']:.4f})")
print("\nDone! Download the JSON files to use for steering.")